# Wan 2.1 Text-to-Video
Google Colab setup para geração de vídeos com ComfyUI

In [7]:
# Setup ComfyUI
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q av

/content
fatal: destination path 'ComfyUI' already exists and is not an empty directory.
/content/ComfyUI


In [8]:
# Install Wan 2.1 nodes
import subprocess, os
os.chdir('/content/ComfyUI/custom_nodes')
try:
  subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/kijai/ComfyUI-Wan.git'], check=True)
  print('Installed via git clone')
except:
  os.system('wget -q https://github.com/kijai/ComfyUI-Wan/archive/refs/heads/main.zip && unzip -q main.zip && mv ComfyUI-Wan-main ComfyUI-Wan')
  print('Installed via ZIP')

Installed via ZIP


In [9]:
# Download models (1.3B version)
os.chdir('/content/ComfyUI')
os.makedirs('models/unet', exist_ok=True)
os.makedirs('models/vae', exist_ok=True)
os.makedirs('models/clip', exist_ok=True)
!wget -q https://huggingface.co/Comfy-Org/Wan_2.1_T2V_1_3B-BP/resolve/main/split_files/diffusion_models/wan2.1_t2v_1.3B_bf16.safetensors -O models/unet/wan2.1_t2v_1.3B_bf16.safetensors
!wget -q https://huggingface.co/Comfy-Org/Wan_2.1_T2V_1_3B-BP/resolve/main/split_files/vae/wan_2.1_vae.safetensors -O models/vae/wan_2.1_vae.safetensors
!wget -q https://huggingface.co/Comfy-Org/Wan_2.1_T2V_1_3B-BP/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors -O models/clip/umt5_xxl_fp8_e4m3fn_scaled.safetensors
print('Models downloaded')

Models downloaded


In [10]:
# Save workflow
wf = '{"last_node_id":47,"last_link_id":93,"nodes":[{"id":38,"type":"CLIPLoader","pos":[12,184],"size":[390,82],"widgets_values":["umt5_xxl_fp8_e4m3fn_scaled.safetensors","wan","default"]},{"id":37,"type":"UNETLoader","pos":[485,57],"size":[346,82],"widgets_values":["wan2.1_t2v_1.3B_bf16.safetensors","default"]},{"id":39,"type":"VAELoader","pos":[866,499],"size":[306,58],"widgets_values":["wan_2.1_vae.safetensors"]},{"id":40,"type":"EmptyHunyuanLatentVideo","pos":[520,620],"size":[315,130],"widgets_values":[832,480,33,1]},{"id":6,"type":"CLIPTextEncode","pos":[415,186],"size":[422,164],"widgets_values":["a fox moving quickly in a beautiful winter scenery nature trees sunset tracking camera"],"color":"#232","bgcolor":"#353"},{"id":7,"type":"CLIPTextEncode","pos":[413,389],"size":[425,180],"widgets_values":["Overexposure, static, blurred details, subtitles, paintings, pictures, still, overall gray"],"color":"#322","bgcolor":"#533"},{"id":3,"type":"KSampler","pos":[863,187],"size":[315,262],"widgets_values":[878361741413693,"randomize",30,6,"uni_pc","simple",1]},{"id":8,"type":"VAEDecode","pos":[1210,190],"size":[210,46]},{"id":47,"type":"SaveWEBM","pos":[2367,193],"size":[315,130],"widgets_values":["ComfyUI","vp9",24,32]},{"id":28,"type":"SaveAnimatedWEBP","pos":[1460,190],"size":[870,643],"widgets_values":["ComfyUI",16,false,90,"default"]}],"links":[[35,3,0,8,0,"LATENT"],[46,6,0,3,1,"CONDITIONING"],[52,7,0,3,2,"CONDITIONING"],[56,8,0,28,0,"IMAGE"],[74,38,0,6,0,"CLIP"],[75,38,0,7,0,"CLIP"],[76,39,0,8,1,"VAE"],[91,40,0,3,3,"LATENT"],[92,37,0,3,0,"MODEL"],[93,8,0,47,0,"IMAGE"]]}'
open('/content/ComfyUI/workflow.json', 'w').write(wf)
print('Workflow saved')

Workflow saved


## Start ComfyUI with LocalTunnel

In [11]:
# Install localtunnel
!npm install -g localtunnel 2>/dev/null


changed 22 packages in 727ms

3 packages are looking for funding
  run `npm fund` for details


In [13]:
# Start ComfyUI and LocalTunnel
import subprocess, os, time, threading
from IPython.display import display, HTML

os.chdir('/content/ComfyUI')

# Start ComfyUI
comfy = subprocess.Popen(['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'])
print('Starting ComfyUI on port 8188...')
time.sleep(10)

# Start localtunnel in background
def tunnel():
  proc = subprocess.Popen(['lt', '--port', '8188'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
  for line in iter(proc.stdout.readline, b''):
    line = line.decode()
    if 'https://' in line:
      print(f'\n🌐 LocalTunnel URL: {line.strip()}')

t = threading.Thread(target=tunnel, daemon=True)
t.start()
time.sleep(3)

display(HTML('<h3>ComfyUI rodando! Aguarde a URL do LocalTunnel acima.</h3><p>Copie a URL que começa com <code>https://</code> e cole no navegador.</p>'))

Starting ComfyUI on port 8188...
